## Random Forest

O [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) é um modelo de ensemble que combina múltiplas árvores de decisão para melhorar a performance e reduzir o overfitting.

Cada árvore é treinada em uma amostra diferente dos dados, e a decisão final é feita por votação.

Esse modelo tende a ser mais robusto e apresentar melhor generalização em comparação com uma única árvore.

### Hiperparâmetros utilizados

- **n_estimators**: número de árvores na floresta.
  - Mais árvores → melhor desempenho (até certo ponto)

- **max_depth**: profundidade máxima das árvores.
  - Controla o overfitting

- **min_samples_split**: mínimo de amostras para dividir um nó

- **min_samples_leaf**: O número mínimo de amostras necessário para que um nó seja uma folha

### Estratégia

Foi utilizado GridSearchCV para encontrar a melhor combinação de hiperparâmetros, utilizando validação cruzada e a métrica ROC AUC.

In [1]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    import pandas as pd
    import sys
    import os

    # add raiz do projeto
    sys.path.append(os.path.abspath(".."))

    from sklearn.model_selection import GridSearchCV
    from imblearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTE
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

    from preprocessing.main_preprocessing import load_and_preprocess
    from utils.experiment_logger import save_results


save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade

🔎 Scenario: submodalidade_agrupada

🔎 Scenario: submodalidade_engineered


""


## BASELINE

In [2]:
# scenarios = [
#     "sem_submodalidade",
#     "submodalidade_agrupada",
#     "submodalidade_engineered"
# ]

# smote_options = [False, True]

# results = []


# for scenario in scenarios:
#     for use_smote in smote_options:

#         print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

#         X_train, X_test, y_train, y_test = load_and_preprocess(
#             "../predictive_models/scrdata_202505.csv",
#             scenario=scenario,
#             use_smote=use_smote
#         )

#         steps = []

#         if use_smote:
#             steps.append(('smote', SMOTE(random_state=42)))
#         steps.append(('rf', RandomForestClassifier(random_state=42, class_weight="balanced")))

#         model = Pipeline(steps)

#         model.fit(X_train, y_train)

#         y_pred = model.predict(X_test)
#         y_proba = model.predict_proba(X_test)[:, 1]

#         results.append({
#             "model": "RandomForest",
#             "scenario": scenario,
#             "smote": use_smote,
#             "roc_auc": roc_auc_score(y_test, y_proba),
#             "f1": f1_score(y_test, y_pred),
#             "accuracy": accuracy_score(y_test, y_pred),
#             "n_features": X_train.shape[1],
#             "phase": "baseline",
#             "timestamp": pd.Timestamp.now()
#         })

#save_results(results, file_path="../results/experiments.csv")

# df_result = pd.DataFrame(results)

# display(df_result)


## GRIDSEARCH

In [3]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    X_train, X_test, y_train, y_test = load_and_preprocess(
        "../predictive_models/scrdata_202505.csv",
        scenario=scenario,
        use_smote=False
    )
    print(f"Colunas utilizadas para o cenario '{scenario}': {X_train.columns.tolist()}")
    print(f"Total de features: {len(X_train.columns)}")


    param_grid_rf = {
        "smote": [SMOTE(random_state=42), "passthrough"],
        "rf__n_estimators": [100, 200],
        "rf__min_samples_split": [2, 5],
        "rf__min_samples_leaf": [1, 2]
    }

    grid_rf = GridSearchCV(
        estimator=Pipeline([
            ('smote', SMOTE(random_state=42)),
            ('rf', RandomForestClassifier(random_state=42,
            n_jobs=2))
        ]),
        param_grid=param_grid_rf,
        cv=5,
        scoring="roc_auc",
        n_jobs=1
    )

    grid_rf.fit(X_train, y_train)

    print("Best parameters:", grid_rf.best_params_)
    print("Best ROC AUC (CV):", grid_rf.best_score_)


    #? BEST MODEL TEST EVALUATION

    best_rf = grid_rf.best_estimator_

    y_pred = best_rf.predict(X_test)
    y_proba = best_rf.predict_proba(X_test)[:, 1]


    #? TUNING (CV)



    cv_results = pd.DataFrame(grid_rf.cv_results_)
    cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

    for smote_val in [True, False]:
        sub_results = cv_results[cv_results['smote_used'] == smote_val]
        if not sub_results.empty:
            best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
            params = best_row['params']

            tuning_results.append({
                "model": "RandomForest",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "tuning_cv",
                "roc_auc": best_row['mean_test_score'],
                "f1": None,
                "accuracy": None,
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

            # Re-fit the best model for this group to get test metrics
            best_model_group = grid_rf.estimator.set_params(**params)
            best_model_group.fit(X_train, y_train)

            y_pred = best_model_group.predict(X_test)
            y_proba = best_model_group.predict_proba(X_test)[:, 1]

            tuning_results.append({
                "model": "RandomForest",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "test",
                "roc_auc": roc_auc_score(y_test, y_proba),
                "f1": f1_score(y_test, y_pred),
                "accuracy": accuracy_score(y_test, y_pred),
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

    save_results(tuning_results, file_path="../results/model_results.csv")

    display(pd.DataFrame(tuning_results))

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade
Colunas utilizadas para o cenario 'sem_submodalidade': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salários mínimos', '

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,RandomForest,sem_submodalidade,True,tuning_cv,0.927304,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:15.806865
1,RandomForest,sem_submodalidade,True,test,0.927406,0.868379,0.849480,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:42.780557
2,RandomForest,sem_submodalidade,False,tuning_cv,0.929177,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:42.785896
3,RandomForest,sem_submodalidade,False,test,0.928845,0.872629,0.851944,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:54:01.401664



🔎 Scenario: submodalidade_agrupada
Colunas utilizadas para o cenario 'submodalidade_agrupada': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salários m

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,RandomForest,sem_submodalidade,True,tuning_cv,0.927304,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:15.806865
1,RandomForest,sem_submodalidade,True,test,0.927406,0.868379,0.849480,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:42.780557
2,RandomForest,sem_submodalidade,False,tuning_cv,0.929177,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:42.785896
3,RandomForest,sem_submodalidade,False,test,0.928845,0.872629,0.851944,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:54:01.401664
4,RandomForest,submodalidade_agrupada,True,tuning_cv,0.943026,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:13:47.494850
5,RandomForest,submodalidade_agrupada,True,test,0.943229,0.884559,0.867202,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:14:19.573252
6,RandomForest,submodalidade_agrupada,False,tuning_cv,0.944314,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:14:19.580542
7,RandomForest,submodalidade_agrupada,False,test,0.944419,0.887914,0.869447,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:14:41.516486



🔎 Scenario: submodalidade_engineered
Colunas utilizadas para o cenario 'submodalidade_engineered': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salári

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,RandomForest,sem_submodalidade,True,tuning_cv,0.927304,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:15.806865
1,RandomForest,sem_submodalidade,True,test,0.927406,0.868379,0.849480,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:42.780557
2,RandomForest,sem_submodalidade,False,tuning_cv,0.929177,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:42.785896
3,RandomForest,sem_submodalidade,False,test,0.928845,0.872629,0.851944,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:54:01.401664
4,RandomForest,submodalidade_agrupada,True,tuning_cv,0.943026,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:13:47.494850
5,RandomForest,submodalidade_agrupada,True,test,0.943229,0.884559,0.867202,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:14:19.573252
6,RandomForest,submodalidade_agrupada,False,tuning_cv,0.944314,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:14:19.580542
7,RandomForest,submodalidade_agrupada,False,test,0.944419,0.887914,0.869447,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:14:41.516486
8,RandomForest,submodalidade_engineered,True,tuning_cv,0.938267,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:32:21.260946
9,RandomForest,submodalidade_engineered,True,test,0.938458,0.878402,0.860376,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:32:49.909985


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,RandomForest,sem_submodalidade,True,tuning_cv,0.927304,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:15.806865
1,RandomForest,sem_submodalidade,True,test,0.927406,0.868379,0.849480,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:42.780557
2,RandomForest,sem_submodalidade,False,tuning_cv,0.929177,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:53:42.785896
3,RandomForest,sem_submodalidade,False,test,0.928845,0.872629,0.851944,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 17:54:01.401664
4,RandomForest,submodalidade_agrupada,True,tuning_cv,0.943026,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:13:47.494850
5,RandomForest,submodalidade_agrupada,True,test,0.943229,0.884559,0.867202,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:14:19.573252
6,RandomForest,submodalidade_agrupada,False,tuning_cv,0.944314,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:14:19.580542
7,RandomForest,submodalidade_agrupada,False,test,0.944419,0.887914,0.869447,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:14:41.516486
8,RandomForest,submodalidade_engineered,True,tuning_cv,0.938267,NaN,NaN,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:32:21.260946
9,RandomForest,submodalidade_engineered,True,test,0.938458,0.878402,0.860376,"{'rf__min_samples_leaf': 1, 'rf__min_samples_s...",2026-06-06 18:32:49.909985
